In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')



In [2]:
train_df = pd.read_csv('Public Dataset/public_trip_data.csv')
test_df = pd.read_csv('private dataset/private_trip_data.csv')

In [3]:
leakage_cols = ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e']
train_df = train_df.drop(columns=[col for col in leakage_cols if col in train_df.columns])

In [4]:
train_df

,TripID,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingType,ShippingTypeDescription,Purpose,OutOfPolicy,EntitiyCode,EmployeeNumber,BusinessUnit,HotelNights,NetCosts,HighCarbon
0,1,CN,Beijing,IN,New Delhi,12,Business Class Flight,Customer Visit,No,9000,M1000-M5008,Services,1,900,0
1,3,US,New York,MX,Mexico City,11,First Class Flight,Customer Visit,Yes,6000,M1000-M5030,Sales,2,3600,0
2,6,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Conference/Exhibition,No,9000,M1000-M5069,Services,1,450,0
3,7,DE,Berlin,FR,Paris,12,Business Class Flight,Customer Visit,No,10000,M1000-M5095,Executive Management,1,3450,0
4,8,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Internal Business Trip,No,7000,M1000-M5097,Marketing,2,1500,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65284,87049,AU,Sydney,DE,Frankfurt,10,Economy Flight,Customer Visit,No,6000,M1555-M5540,Sales,4,2400,1
65285,87050,DE,Berlin,ES,Madrid,15,Volkswagen Golf petrol,Customer Visit,No,8000,M1555-M5546,Executive Management,5,2850,0
65286,87051,BR,SÃ£o Paulo,US,Miami,12,Business Class Flight,Customer Visit,Yes,10000,M1555-M5554,Sales,5,2100,1
65287,87052,AU,Sydney,AU,Melbourne,13,BMW 3 diesel,Customer Visit,No,8000,M1555-M5555,Marketing,5,1800,0


In [5]:
target_col = 'HighCarbon'
X = train_df.drop(columns=['TripID', target_col])
y = train_df[target_col]

test_trip_ids = test_df['TripID']
X_test = test_df.drop(columns=['TripID'])

if 'EmployeeNumber' in X.columns and 'EmployeeNumber' not in X_test.columns:
    X = X.drop(columns=['EmployeeNumber'])

X_test = X_test[X.columns]

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = pd.Categorical(X_test[col], categories=X[col].cat.categories)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [6]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=4,
    num_leaves=12,
    min_child_samples=100,
    subsample=0.7,
    colsample_bytree=0.7,
    class_weight='balanced',
    max_bin=64,
    min_data_per_group=100,
    verbose=-1,
    random_state=42
)

model.fit(
    X_train, 
    y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

,boosting_type,'gbdt'
,num_leaves,12
,max_depth,4
,learning_rate,0.01
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


In [7]:
val_probs = model.predict_proba(X_val)[:, 1]
val_preds = model.predict(X_val)

print("--- Validation Metrics ---")
print(f"ROC-AUC Score: {roc_auc_score(y_val, val_probs):.4f}")
print(f"F1 Score:      {f1_score(y_val, val_preds):.4f}")
print(f"Precision:     {precision_score(y_val, val_preds):.4f}")
print(f"Recall:        {recall_score(y_val, val_preds):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))

--- Validation Metrics ---
ROC-AUC Score: 0.9994
F1 Score:      0.9873
Precision:     0.9898
Recall:        0.9847

Confusion Matrix:
[[9760   33]
 [  50 3215]]


In [8]:
test_probs = model.predict_proba(X_test)[:, 1]

submission_df = pd.DataFrame({
    'TripID': test_trip_ids,
    'HighCarbon': test_probs
})

submission_df.to_csv('submission.csv', index=False)

In [9]:
test_probs = model.predict_proba(X_test)[:, 1]

submission_df = pd.DataFrame({
    'TripID': test_trip_ids,
    'HighCarbon': test_probs
})

submission_df.to_csv('submission.csv', index=False)